# D08 -- 拟时序多方法横向比较

本 notebook 对前述拟时序 notebook（10/10b/10c/10d）的产出做横向比较，
回答核心问题：**不同方法给出的 pseudotime / potency 排序是否一致？**

## 设计原则

- **自动扫描可用方法**：遍历 `METHOD_SOURCES` 配置，检查文件是否存在、目标列是否有效、
  方法是否显式跳过。只比较实际可用的方法，缺哪个跳哪个，绝不因某方法缺失而崩。
- **方向对齐**：potency 类方法（CytoTRACE、转录组熵）高分 = 未分化，
  pseudotime 类方法高分 = 分化晚期。比较时对 potency 取反（1 - score），
  使所有分数同向"高分 = 分化晚期"。
- **不崩原则**：任何方法缺失、列不存在、全 NaN、细胞类型列缺失等都不导致 notebook 崩溃，
  打印原因后优雅跳过对应分析。
- **本 notebook 纯读取 + 比较，不做上皮 subset、不做 root 识别、不调 R/外部工具**。


## 输入与输出

| 项目 | 路径/字段 |
|------|-----------|
| 方法来源 | `METHOD_SOURCES` 字典（见下方 PARAMS cell），指向 `results/10*_v1.h5ad` |
| 输出 h5ad | `OUTPUT_PATH`（默认 `results/10e_pseudotime_compare_v1.h5ad`）|
| 输出 CSV | `results/10e_pseudotime_method_comparison.csv` |
| Spearman CSV | `results/10e_spearman_correlation.csv` |
| Figure 目录 | `results/figures/`（`10e_spearman_correlation.png`、`10e_method_scores_umap.png` 等）|

## 与上下游 notebook 的关系

| Notebook | 角色 | 产物 |
|----------|------|------|
| `D04_pseudotime.ipynb` | 综合拟时序（熵 + CytoTRACE v1 + root + Monocle3）| `results/10_pseudotime_v1.h5ad` |
| `D05_pseudotime_monocle3.ipynb` | Monocle3 独立版 | `results/10b_pseudotime_monocle3_v1.h5ad` |
| `D06_pseudotime_cellrank2.ipynb` | CellRank2 | `results/10c_pseudotime_cellrank2_v1.h5ad` |
| `D07_potency_cytotrace2.ipynb` | CytoTRACE2 | `results/10d_pseudotime_cytotrace2_v1.h5ad` |
| **`D08_pseudotime_compare.ipynb`（本 notebook）** | **多方法横向比较** | `results/10e_*` |


In [ ]:
# === PARAMS ===
# METHOD_SOURCES -- 各拟时序方法的产物路径 + 关键 obs 列 + 类型
#   type: "pseudotime" (高分=分化晚期) 或 "potency" (高分=未分化，需取反对齐)
#   pseudotime_col: obs 中的 pseudotime 列名
#   potency_col: obs 中的 potency 列名
#   fate_obsm: obsm 中的 fate probability 矩阵 key（可选，仅用于可视化注释）

METHOD_SOURCES = {
    "monocle3": {
        "path": "results/10b_pseudotime_monocle3_v1.h5ad",
        "pseudotime_col": "pseudotime_monocle3_v1",
        "type": "pseudotime",
    },
    "monocle3_combined": {
        "path": "results/10_pseudotime_v1.h5ad",
        "pseudotime_col": "pseudotime_monocle3_v1",
        "type": "pseudotime",
    },
    "cytotrace_v1": {
        "path": "results/10_pseudotime_v1.h5ad",
        "potency_col": "cytotrace_score",
        "type": "potency",
    },
    "entropy": {
        "path": "results/10_pseudotime_v1.h5ad",
        "potency_col": "entropy",
        "type": "potency",
    },
    "cellrank2_cytotrace": {
        "path": "results/10c_pseudotime_cellrank2_v1.h5ad",
        "potency_col": "cellrank2_cytotrace",
        "type": "potency",
        "fate_obsm": "cellrank2_fate_probabilities",
    },
    "cellrank2_ct_pseudotime": {
        "path": "results/10c_pseudotime_cellrank2_v1.h5ad",
        "pseudotime_col": "cellrank2_pseudotime",  # 该列仅在 GPCCA 成功且产出 pseudotime 属性时存在，否则扫描阶段正确跳过
        "type": "pseudotime",
    },
    "cytotrace2": {
        "path": "results/10d_pseudotime_cytotrace2_v1.h5ad",
        "potency_col": "cytotrace2_potency_score",
        "type": "potency",
    },
}

OUTPUT_PATH = "results/10e_pseudotime_compare_v1.h5ad"
COMPARE_FIGDIR = "results/figures"
COMPARE_CSV_PATH = "results/tables/10e_pseudotime_method_comparison.csv"
SPEARMAN_CSV_PATH = "results/tables/10e_spearman_correlation.csv"
CELL_TYPE_COL = "cell_type_final_v1"  # 不存在或全 NaN 时优雅跳过

In [ ]:
# === setup: sys.path + env_check + imports + cd to project root ===
import sys, os, gc

_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")

# 环境自检（与其他 notebook 一致的 env_check 模式）
try:
    from scrna_integration.platform import env_check
    env_check(expected_env="scrna-integration")
except Exception as _e:
    print(f"环境自检跳过（env_check 不可用: {_e}）")

# 导入依赖
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

print(f"scanpy {sc.__version__}  |  numpy {np.__version__}  |  seaborn {sns.__version__}")

In [ ]:
# === 扫描可用方法 ===
# 对 METHOD_SOURCES 中每个方法检查：文件存在？可读？目标列在 obs 且非全 NaN？
# 记录可用方法到 _available_methods，跳过的方法到 _skip_reasons。

_available_methods = {}   # name -> {path, col, type, n_cells, obs_names, _adata}
_skip_reasons = {}        # name -> [reasons...]
_adata_cache = {}         # path -> adata (避免重复读取同文件)

for _method_name, _cfg in METHOD_SOURCES.items():
    _fpath = _cfg["path"]
    _reasons = []

    # 1. 文件存在？
    if not os.path.exists(_fpath):
        _reasons.append(f"文件不存在: {_fpath}")
        _skip_reasons[_method_name] = _reasons
        continue

    # 2. 可读？（复用已读缓存）
    if _fpath in _adata_cache:
        _adata_tmp = _adata_cache[_fpath]
    else:
        try:
            _adata_tmp = sc.read_h5ad(_fpath)
            _adata_cache[_fpath] = _adata_tmp
        except Exception as _e:
            _reasons.append(f"读取失败: {_e}")
            _skip_reasons[_method_name] = _reasons
            continue

    # 3. 确定目标列
    if "pseudotime_col" in _cfg:
        _col = _cfg["pseudotime_col"]
        _col_label = "pseudotime_col"
    elif "potency_col" in _cfg:
        _col = _cfg["potency_col"]
        _col_label = "potency_col"
    else:
        _reasons.append("METHOD_SOURCES 配置缺少 pseudotime_col 或 potency_col")
        _skip_reasons[_method_name] = _reasons
        continue

    # 4. 列存在？
    if _col not in _adata_tmp.obs.columns:
        # 尝试从 uns 获取更详细的跳过原因（如 *_ran=False）
        _extra = ""
        for _uk in _adata_tmp.uns.keys():
            if isinstance(_adata_tmp.uns[_uk], dict):
                for _rk in _adata_tmp.uns[_uk]:
                    if "ran" in _rk.lower() and not _adata_tmp.uns[_uk][_rk]:
                        _extra = f"（{_uk}['{_rk}']=False）"
                        break
            if _extra:
                break
        _reasons.append(f"obs 列 '{_col}' 不存在{_extra}")
        _skip_reasons[_method_name] = _reasons
        continue

    # 5. 列全 NaN？
    _col_data = _adata_tmp.obs[_col]
    if _col_data.isna().all():
        _nan_pct = _col_data.isna().mean() * 100
        _reasons.append(f"obs 列 '{_col}' 全为 NaN（{_nan_pct:.0f}%，{_col_label}）")
        _skip_reasons[_method_name] = _reasons
        continue

    # 通过所有检查 -- 方法可用
    _info = {
        "path": _fpath,
        "col": _col,
        "type": _cfg.get("type", "unknown"),
        "n_cells": _adata_tmp.n_obs,
        "obs_names": set(_adata_tmp.obs_names),
        "_adata": _adata_tmp,
    }
    if "fate_obsm" in _cfg:
        _info["fate_obsm"] = _cfg["fate_obsm"]
    _available_methods[_method_name] = _info

# --- 打印扫描结果 ---
print("=" * 60)
print("拟时序方法可用性扫描")
print("=" * 60)
print(f"\n配置方法总数: {len(METHOD_SOURCES)}")
print(f"可用方法: {len(_available_methods)}")
print(f"跳过/缺失: {len(_skip_reasons)}")

if _available_methods:
    print(f"\n可用方法详情:")
    for _name, _info in _available_methods.items():
        _extra = ""
        if _info.get("fate_obsm"):
            _extra = f", fate_obsm={_info['fate_obsm']}"
        _n_cells = _info['n_cells']
        print(f"  [{_info['type']:11s}] {_name:30s} -> {_info['col']} "
              f"({_n_cells:,} cells{_extra})")
else:
    print(f"\n可用方法: （无）")

if _skip_reasons:
    print(f"\n跳过/缺失方法详情:")
    for _name, _reasons in _skip_reasons.items():
        print(f"  {_name}: {'; '.join(_reasons)}")
else:
    print(f"\n跳过/缺失方法: （无）")
print("=" * 60)


In [ ]:
# === 对齐细胞 ===
# 各方法产物可能细胞数不同（如上皮 subset），取交集 obs_names 作为公共比较集。
# 构建 _score_df_raw（原始分数）和 _score_df_aligned（方向对齐后，高分=分化晚期）。

# 初始化空变量，确保 skip 路径下下游 cell 不崩
_common_cells = []
_score_df_raw = pd.DataFrame()
_score_df_aligned = pd.DataFrame()

if len(_available_methods) == 0:
    print("无可用的拟时序方法，跳过细胞对齐。后续分析全部跳过。")
else:
    # 取交集 obs_names
    _common_sets = None
    for _name, _info in _available_methods.items():
        if _common_sets is None:
            _common_sets = _info["obs_names"]
        else:
            _common_sets = _common_sets & _info["obs_names"]
    _common_cells = sorted(_common_sets)

    print(f"各方法细胞数及与公共交集的对比:")
    for _name, _info in _available_methods.items():
        _n_total = _info["n_cells"]
        _n_common = len(_info["obs_names"] & _common_sets)
        _pct = _n_common / _n_total * 100 if _n_total > 0 else 0
        print(f"  {_name:30s}: {_n_total:>6,} total, {_n_common:>6,} in common "
              f"({_pct:.1f}%)")
    print(f"\n公共细胞交集: {len(_common_cells):,} cells")

    # 构建 _score_df_raw（原始分数）和 _score_df_aligned（方向对齐后）
    _score_df_raw = pd.DataFrame(index=_common_cells)
    _score_df_aligned = pd.DataFrame(index=_common_cells)

    for _name, _info in _available_methods.items():
        _adata_src = _info["_adata"]
        _col = _info["col"]
        _type = _info["type"]

        # 取公共细胞的分数（to_numeric 处理非数值如字符串 "nan"）
        _scores = pd.to_numeric(
            _adata_src[_common_cells].obs[_col], errors="coerce"
        )
        _score_df_raw[_name] = _scores.values

        # 方向对齐: potency 取反 (高分=未分化 -> 高分=分化晚期)
        # min-max 归一化后再取反，确保对齐值始终落在 [0, 1]，避免 entropy 等非
        # [0,1] 范围分数经 1-score 后产生负值导致 UMAP 色阶判读困惑
        if _type == "potency":
            _vals = _scores.values
            _vmin, _vmax = np.nanmin(_vals), np.nanmax(_vals)
            _eps = 1e-12
            _normed = (_vals - _vmin) / (_vmax - _vmin + _eps)
            _score_df_aligned[_name] = 1.0 - _normed
            print(f"  {_name}: potency -> min-max 归一化后取反对齐，"
                  f"range=[{_score_df_aligned[_name].min():.4f}, "
                  f"{_score_df_aligned[_name].max():.4f}]")
        else:
            _score_df_aligned[_name] = _scores.values
            print(f"  {_name}: pseudotime -> 保持原方向，"
                  f"range=[{_score_df_aligned[_name].min():.4f}, "
                  f"{_score_df_aligned[_name].max():.4f}]")

    # 检查交集后全 NaN 列：某方法自身有有效值但公共细胞上全 NaN，
    # 会导致 Spearman 矩阵出 NaN 列。检测到后从 _score_df 和 _available_methods 移除
    _nan_ratio = _score_df_aligned.isna().mean()
    _all_nan_cols = _nan_ratio[_nan_ratio >= 1.0].index.tolist()
    if _all_nan_cols:
        print(f"\n以下方法在公共细胞交集上全为 NaN，已从后续分析移除：")
        for _col_name in _all_nan_cols:
            print(f"  {_col_name}: NaN 占比 100%，公共细胞上无有效值")
        _score_df_aligned = _score_df_aligned.drop(columns=_all_nan_cols)
        _score_df_raw = _score_df_raw.drop(columns=_all_nan_cols)
        for _col_name in _all_nan_cols:
            _available_methods.pop(_col_name, None)
        print(f"移除后剩余 {len(_available_methods)} 个方法")

    # 检查对齐后 NaN
    _nan_counts = _score_df_aligned.isna().sum()
    if _nan_counts.sum() > 0:
        print(f"\nWARNING: 对齐后存在 NaN 值的列:")
        for _col_name in _nan_counts[_nan_counts > 0].index:
            print(f"  {_col_name}: {_nan_counts[_col_name]} NaN cells")

    print(f"\n_score_df_aligned.shape: {_score_df_aligned.shape}")
    print(f"\n对齐后分数概况 (aligned: higher = later differentiation):")
    print(_score_df_aligned.describe().round(4).to_string())

In [ ]:
# === Spearman 秩相关矩阵 ===
# 计算各方法对齐后分数的两两 Spearman 秩相关，画热图。

_spearman_corr = pd.DataFrame()

_n_methods = len(_available_methods)
if _n_methods < 2:
    print(f"可用方法数 = {_n_methods}，不足 2 个，无法计算相关矩阵。")
    if _n_methods == 1:
        _only_name = list(_available_methods.keys())[0]
        _only_scores = _score_df_aligned[_only_name].dropna()
        print(f"仅 {_only_name} 可用，"
              f"有效细胞数 = {len(_only_scores)}/{len(_common_cells)}")
elif _score_df_aligned.empty or _score_df_aligned.shape[1] < 2:
    print("_score_df_aligned 为空或仅 1 列，无法计算相关矩阵。")
else:
    _spearman_corr = _score_df_aligned.corr(method="spearman")
    print("Spearman 秩相关矩阵（方向对齐后，高分=分化晚期）:")
    print(_spearman_corr.round(4).to_string())

    # 画热图
    _fig, _ax = plt.subplots(
        figsize=(max(6, _n_methods * 1.3), max(5, _n_methods * 1.1))
    )
    _mask = np.triu(np.ones_like(_spearman_corr, dtype=bool), k=1)
    sns.heatmap(
        _spearman_corr, annot=True, fmt=".3f", cmap="RdBu_r",
        vmin=-1, vmax=1, center=0, square=True,
        mask=_mask, linewidths=0.5, ax=_ax,
        cbar_kws={"label": "Spearman rho", "shrink": 0.8},
    )
    _ax.set_title(
        "Pseudotime/Potency Method Spearman Correlation\n"
        "(aligned direction: higher = later differentiation)",
        fontsize=11,
    )
    _fig.tight_layout()
    _heatmap_path = os.path.join(COMPARE_FIGDIR, "10e_spearman_correlation.png")
    _fig.savefig(_heatmap_path, dpi=150, bbox_inches="tight")
    print(f"\n热图已保存: {_heatmap_path}")
    plt.show()


In [ ]:
# === Root/起点一致性 ===
# 比较不同 potency 方法识别的"最不分化细胞"是否一致。
# 对每个 potency 方法取 top 5% 最未分化细胞（原始分数最高 = 最未分化），
# 计算两两 Jaccard 重叠系数。

_potency_methods = {
    k: v for k, v in _available_methods.items() if v["type"] == "potency"
}
_n_potency = len(_potency_methods)

if _n_potency < 2:
    print(f"Potency 方法数 = {_n_potency}，不足 2 个，跳过 root/起点一致性分析。")
    if _n_potency == 1:
        _only_name = list(_potency_methods.keys())[0]
        print(f"仅 {_only_name} 为 potency 类型，"
              f"无法与其他 potency 方法比较起点。")
elif len(_common_cells) == 0:
    print("公共细胞交集为空，跳过 root 一致性分析。")
else:
    _top_pct = 0.05
    _top_cells = {}

    for _name, _info in _potency_methods.items():
        _adata_src = _info["_adata"]
        _col = _info["col"]
        # 取公共细胞的原始 potency 分数（高分=未分化）
        _scores = pd.to_numeric(
            _adata_src[_common_cells].obs[_col], errors="coerce"
        ).dropna()

        if len(_scores) == 0:
            print(f"  {_name}: 所有公共细胞分数为 NaN，跳过")
            _top_cells[_name] = set()
            continue

        _threshold = _scores.quantile(1 - _top_pct)
        _top_set = set(_scores[_scores >= _threshold].index.tolist())
        _top_cells[_name] = _top_set
        print(f"  {_name}: top {_top_pct*100:.0f}% threshold={_threshold:.4f}, "
              f"cells={len(_top_set)}/{len(_scores)} "
              f"({len(_top_set)/max(len(_scores),1)*100:.1f}%)")

    # 两两 Jaccard 重叠
    _method_names = sorted(_potency_methods.keys())
    _pairs_found = False
    print(f"\n两两 Jaccard 重叠（top {_top_pct*100:.0f}% 最未分化细胞）:")

    for _i, _n1 in enumerate(_method_names):
        for _j, _n2 in enumerate(_method_names):
            if _j <= _i:
                continue
            _s1, _s2 = _top_cells.get(_n1, set()), _top_cells.get(_n2, set())
            if not _s1 or not _s2:
                print(f"  {_n1} vs {_n2}: 跳过（一方无有效细胞）")
                continue
            _intersection = _s1 & _s2
            _union = _s1 | _s2
            _jaccard = len(_intersection) / len(_union) if _union else 0.0
            _pairs_found = True
            print(f"  {_n1} vs {_n2}: Jaccard={_jaccard:.4f} "
                  f"(intersection={len(_intersection)}, union={len(_union)})")

    if not _pairs_found:
        print("  （无有效方法对）")


In [ ]:
# === UMAP 并排可视化 ===
# 如果某个产物有 X_umap，把各方法的对齐后 score 并排画在 UMAP 上。

_has_umap = False
_umap_adata = None
_umap_source_name = None

for _name, _info in _available_methods.items():
    if "X_umap" in _info["_adata"].obsm:
        _has_umap = True
        _umap_adata = _info["_adata"]
        _umap_source_name = _name
        break

if not _has_umap:
    print("所有产物均无 X_umap，跳过 UMAP 并排可视化。")
elif len(_available_methods) == 0:
    print("无可用的拟时序方法，跳过 UMAP 可视化。")
elif len(_common_cells) == 0:
    print("公共细胞交集为空，跳过 UMAP 可视化。")
else:
    print(f"使用 {_umap_source_name} 的 X_umap "
          f"({_umap_adata.obsm['X_umap'].shape})")

    _n_plots = len(_available_methods)
    _n_cols = min(3, _n_plots)
    _n_rows = int(np.ceil(_n_plots / _n_cols))

    _fig, _axes = plt.subplots(
        _n_rows, _n_cols,
        figsize=(5.5 * _n_cols, 4.5 * _n_rows),
    )
    if _n_plots == 1:
        _axes = [np.atleast_1d(_axes)[0]]
    else:
        _axes = _axes.flatten() if hasattr(_axes, "flatten") else [_axes]

    # 获取公共细胞的 UMAP 坐标
    _umap_coords = _umap_adata[_common_cells].obsm["X_umap"]

    for _idx, (_name, _info) in enumerate(_available_methods.items()):
        _ax = _axes[_idx]
        _scores = _score_df_aligned[_name].values
        _valid = ~np.isnan(_scores)
        _sc = _ax.scatter(
            _umap_coords[_valid, 0], _umap_coords[_valid, 1],
            c=_scores[_valid], cmap="viridis", s=2, alpha=0.7,
            rasterized=True,
        )
        plt.colorbar(_sc, ax=_ax, label="aligned score", shrink=0.8)
        _type_label = _info["type"]
        _ax.set_title(f"{_name}\n({_type_label}, aligned)", fontsize=10)
        _ax.set_xlabel("UMAP1")
        _ax.set_ylabel("UMAP2")

    # 隐藏多余子图
    for _idx in range(_n_plots, len(_axes)):
        _axes[_idx].set_visible(False)

    _fig.suptitle(
        "Method Scores on UMAP (aligned: higher = later differentiation)",
        fontsize=13, y=1.01,
    )
    _fig.tight_layout()
    _umap_path = os.path.join(COMPARE_FIGDIR, "10e_method_scores_umap.png")
    _fig.savefig(_umap_path, dpi=150, bbox_inches="tight")
    print(f"UMAP 并排图已保存: {_umap_path}")
    plt.show()


In [ ]:
# === 按细胞类型分组比较 ===
# 检查 CELL_TYPE_COL 是否存在且非全 NaN，然后画各方法 score 在细胞类型上的分布。

if len(_available_methods) == 0:
    print("无可用的拟时序方法，跳过按细胞类型分组比较。")
elif len(_common_cells) == 0:
    print("公共细胞交集为空，跳过按细胞类型分组比较。")
else:
    # 找一个有 CELL_TYPE_COL 的 adata
    _ct_available = False
    _ct_source_adata = None
    for _name, _info in _available_methods.items():
        if CELL_TYPE_COL in _info["_adata"].obs.columns:
            _ct_col = _info["_adata"].obs[CELL_TYPE_COL]
            # 检查是否全 NaN 或全相同
            _non_na = _ct_col.dropna()
            if len(_non_na) > 0 and _non_na.nunique() > 1:
                _ct_available = True
                _ct_source_adata = _info["_adata"]
                break

    if not _ct_available:
        # 给出具体原因
        _col_in_any = any(
            CELL_TYPE_COL in _info["_adata"].obs.columns
            for _info in _available_methods.values()
        )
        if _col_in_any:
            _first_with = next(
                _info for _info in _available_methods.values()
                if CELL_TYPE_COL in _info["_adata"].obs.columns
            )
            _non_na = _first_with["_adata"].obs[CELL_TYPE_COL].dropna()
            if len(_non_na) == 0:
                print(f"CELL_TYPE_COL='{CELL_TYPE_COL}' 存在但全为 NaN，"
                      f"跳过按细胞类型分组比较。")
            else:
                print(f"CELL_TYPE_COL='{CELL_TYPE_COL}' 仅 "
                      f"{_non_na.nunique()} 个唯一非 NaN 值，不足以分组比较，"
                      f"跳过。")
        else:
            print(f"CELL_TYPE_COL='{CELL_TYPE_COL}' 在所有产物中不存在，"
                  f"跳过按细胞类型分组比较。")
    else:
        # 取公共细胞的细胞类型标签
        _ct_labels = _ct_source_adata[_common_cells].obs[CELL_TYPE_COL]
        # 先转为 object 再 fillna，避免 Categorical 新类别 TypeError
        _ct_labels = _ct_labels.astype(object).fillna("Unknown").astype(str)
        _ct_counts = _ct_labels.value_counts()
        _n_ct = _ct_labels.nunique()

        print(f"细胞类型分布 ({CELL_TYPE_COL}, {_n_ct} 类型):")
        for _ct, _cnt in _ct_counts.items():
            print(f"  {_ct}: {_cnt:>6,} cells")

        # 箱线图：每个方法 x 细胞类型
        _n_methods = len(_available_methods)
        _fig, _axes = plt.subplots(
            _n_methods, 1,
            figsize=(max(8, _n_ct * 0.7), 4.2 * _n_methods),
        )
        if _n_methods == 1:
            _axes = [_axes]

        for _idx, (_name, _info) in enumerate(_available_methods.items()):
            _ax = _axes[_idx]
            _plot_df = pd.DataFrame({
                "score": _score_df_aligned[_name].values,
                "cell_type": _ct_labels.values,
            })
            # 按中位数排序细胞类型
            _order = (
                _plot_df.groupby("cell_type")["score"].median()
                .sort_values().index.tolist()
            )
            sns.boxplot(
                data=_plot_df, x="cell_type", y="score",
                order=_order, palette="Set3", ax=_ax, fliersize=1,
            )
            _ax.set_title(
                f"{_name} ({_info['type']}, aligned)", fontsize=11,
            )
            _ax.set_xlabel("")
            _ax.tick_params(axis="x", rotation=45, labelsize=8)

        _fig.suptitle(
            "Method Scores by Cell Type "
            "(aligned: higher = later differentiation)",
            fontsize=13, y=1.01,
        )
        _fig.tight_layout()
        _ct_path = os.path.join(COMPARE_FIGDIR, "10e_scores_by_celltype.png")
        _fig.savefig(_ct_path, dpi=150, bbox_inches="tight")
        print(f"\n细胞类型箱线图已保存: {_ct_path}")
        plt.show()

In [ ]:
# === 汇总结论 ===
# 打印 markdown 式总结，供 PI 判读。

print("## D08 拟时序多方法横向比较 -- 汇总结论\n")

_n_total = len(METHOD_SOURCES)
_n_avail = len(_available_methods)
_n_skip = len(_skip_reasons)

print(f"**扫描结果**：METHOD_SOURCES 共配置 {_n_total} 个方法，"
      f"实际可用 {_n_avail} 个，跳过 {_n_skip} 个。\n")

if _skip_reasons:
    print("**缺失方法及原因**：")
    for _name, _reasons in _skip_reasons.items():
        print(f"- `{_name}`: {'; '.join(_reasons)}")
    print()

if _n_avail == 0:
    print("**结论**：当前所有拟时序方法均不可用，无法进行横向比较。\n"
          "请先成功运行 10/10b/10c/10d 中至少 2 个 notebook，"
          "产生有效的 pseudotime/potency 列后再回看本 notebook。")
elif _n_avail == 1:
    _only_name = list(_available_methods.keys())[0]
    print(f"**结论**：仅 1 个方法可用 (`{_only_name}`)，"
          f"无法进行跨方法比较。\n"
          f"建议至少成功运行 2 个独立的拟时序方法后再看横向一致性。"
          f"注意：来自同一上游文件的多个分数 "
          f"(如 cytotrace 与 ct_pseudotime) 是数学变换关系，"
          f"不代表独立的交叉验证。")
else:
    # 构建 method_name -> upstream path 映射，用于检测同源方法对
    _method_paths = {
        _name: _info["path"]
        for _name, _info in _available_methods.items()
    }

    # 相关矩阵概览
    if not _spearman_corr.empty:
        print("**两两 Spearman 相关概览**（方向对齐后，高分=分化晚期）:\n")

        _pairs = []
        for _i, _n1 in enumerate(_spearman_corr.columns):
            for _j, _n2 in enumerate(_spearman_corr.columns):
                if _j <= _i:
                    continue
                _r = _spearman_corr.loc[_n1, _n2]
                _pairs.append((_n1, _n2, _r))
        _pairs.sort(key=lambda x: x[2], reverse=True)

        print("| 方法 A | 方法 B | Spearman rho | 解读 |")
        print("|--------|--------|-------------|------|")
        for _a, _b, _r in _pairs:
            if _r >= 0.8:
                # 检查是否为同源方法对（同一 h5ad path 派生的不同分数）
                if _method_paths.get(_a) == _method_paths.get(_b):
                    _interp = "高相关（同源上游，非独立验证）"
                else:
                    _interp = "高度一致，可互相印证"
            elif _r >= 0.5:
                _interp = "中等一致"
            elif _r >= 0.3:
                _interp = "弱一致，存在分歧"
            else:
                _interp = "几乎不一致，需重点关注"
            print(f"| `{_a}` | `{_b}` | {_r:.3f} | {_interp} |")
        print()

        # 标记高/低一致性
        _high = [(a, b, r) for a, b, r in _pairs if r >= 0.8]
        _low = [(a, b, r) for a, b, r in _pairs if r < 0.5]
        if _high:
            print("**高一致性方法对**（可互相印证）：")
            for _a, _b, _r in _high:
                _path_tag = ""
                if _method_paths.get(_a) == _method_paths.get(_b):
                    _path_tag = " （同源上游，非独立验证）"
                print(f"- `{_a}` vs `{_b}` (rho={_r:.3f}){_path_tag}")
        if _low:
            print("\n**低一致性方法对**（需关注分歧）：")
            for _a, _b, _r in _low:
                print(f"- `{_a}` vs `{_b}` (rho={_r:.3f})")
        print()

    # Root 一致性
    if "_potency_methods" in dir() and len(_potency_methods) >= 2:
        print("**Root/起点一致性**：见上方 Jaccard 重叠分析。")
        print("高 Jaccard 值 (>0.5) 表明不同 potency 方法在识别"
              "最不分化细胞上相对一致；低值提示潜在的生物学或方法学差异。\n")

print("**判读提示**：")
print("- 不同方法基于不同假设（转录组熵、图拓扑、RNA velocity 等），"
      "排序有分歧是正常的")
print("- 如果多数方法在某段轨迹上一致，该段的生物学解释更可靠")
print("- 建议结合 marker 基因表达、已知生物学知识判断哪种方法的排序更合理")
print("- 如果来自同一上游文件的方法间高相关（如 cytotrace 与 ct_pseudotime），"
      "这是数学变换关系，不代表独立的交叉验证")
print("- 如果有 X_umap 并排图，可直观对比各方法在 embedding 上的分布模式")

In [ ]:
# === 输出 checkpoint ===
# 把 _score_df_aligned 合并到 adata，写 h5ad + CSV。
# provenance 字段与其他 stage notebook 一致。

# 收集所有上游路径（去重）
_upstream_paths = sorted(set(
    _info["path"] for _info in _available_methods.values()
))

if len(_available_methods) == 0:
    print("无可用的拟时序方法，创建最小 adata 仅记录元数据。")
    _out_adata = sc.AnnData(X=sp.csr_matrix((0, 0)))
else:
    # 基于第一个可用 adata subset 公共细胞
    _first_name = list(_available_methods.keys())[0]
    _out_adata = _available_methods[_first_name]["_adata"][_common_cells].copy()

    # 写入多方法 score 到 obs（对齐后 + 原始）
    for _name in _available_methods.keys():
        _out_adata.obs[f"10e_{_name}"] = _score_df_aligned[_name].values
        _out_adata.obs[f"10e_{_name}_raw"] = _score_df_raw[_name].values

    _new_cols = [c for c in _out_adata.obs.columns if c.startswith("10e_")]
    print(f"输出 adata: {_out_adata.n_obs:,} cells x "
          f"{_out_adata.n_vars:,} genes")
    print(f"新增 obs 列 ({len(_new_cols)}): {_new_cols}")

# --- provenance (与其他 stage 一致的嵌套 dict 模式) ---
_out_adata.uns["stage"] = "10e_pseudotime_compare"
_out_adata.uns["version"] = "v1"
_out_adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"
_out_adata.uns["upstream"] = _upstream_paths

# 构建嵌套 uns 字典
_compare_meta = {
    "available_methods": sorted(_available_methods.keys()),
    "skipped_methods": {
        k: v for k, v in sorted(_skip_reasons.items())
    },
    "n_common_cells": len(_common_cells),
    "n_methods_configured": len(METHOD_SOURCES),
    "n_methods_available": len(_available_methods),
    "direction_note": (
        "aligned: higher = later differentiation; "
        "potency scores inverted (1 - score)"
    ),
    "method_sources": {
        k: {
            "path": v["path"],
            "col": v["col"],
            "type": v["type"],
        }
        for k, v in sorted(_available_methods.items())
    },
}
if not _spearman_corr.empty:
    _compare_meta["spearman_correlation"] = {
        k: {kk: round(vv, 6) for kk, vv in v.items()}
        for k, v in _spearman_corr.to_dict().items()
    }

_out_adata.uns["10e_pseudotime_compare_v1"] = _compare_meta

# 写 h5ad
_out_adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"\nh5ad 已写出: {OUTPUT_PATH} "
      f"({os.path.getsize(OUTPUT_PATH):,} bytes)")

# 写方法比较 CSV
if len(_available_methods) > 0:
    _score_df_aligned.to_csv(COMPARE_CSV_PATH)
    print(f"方法比较 CSV 已写出: {COMPARE_CSV_PATH} "
          f"({os.path.getsize(COMPARE_CSV_PATH):,} bytes)")
else:
    pd.DataFrame({"note": ["no methods available"]}).to_csv(
        COMPARE_CSV_PATH, index=False
    )
    print(f"方法比较 CSV 已写出（空）: {COMPARE_CSV_PATH}")

# 写 Spearman 相关矩阵 CSV
if not _spearman_corr.empty:
    _spearman_corr.to_csv(SPEARMAN_CSV_PATH)
    print(f"Spearman 相关矩阵 CSV 已写出: {SPEARMAN_CSV_PATH} "
          f"({os.path.getsize(SPEARMAN_CSV_PATH):,} bytes)")

print("\n10e 多方法横向比较完成。")

# 释放大对象
del _out_adata
gc.collect()
